# Motivation

From the learnings in the `identifiability_and_cramer_rao` crusade, we learned how to compute the rank of the tangent space Jacobian of our model. This allows us to numerically estimate the number of learnable degrees of freedom of the model. In principle, as we are not in one of the "defective" points in the manifold, the rank should be the same. In theory, the exact number should be 


$$\min(3 N, \text{GST}_{\text{min}})$$

with $N$ the number of sequences and $\text{GST}_{\text{min}}$ the theoretical lower bound on the number of parameters after quotienting the Gauge degrees of freedom.

# Goal:

In this notebook, we want to verify a few things:

1. Compare the rank obtained for different points, i.e. the true gate set, the perturbed initialization of the optimization (and perhaps final optimized one)?

2. Verify which of these achieve the theoretical bound.

Importantly, this is only a measurement of learnability in the inifinite shots limit, without any notion of how well this are actually learned in practice. That is what the next crusade will be about, making use of the eigenvalues to compute standard deviations.

In [6]:
%load_ext autoreload
%autoreload 2

from mGST.gauge_analysis import probability_jacobian_spectrum
from mGST.utility_functions_comparisons import get_target_operators_gate_set_name, get_compressed_rep_from_mgst_output, kraus_tensor_to_mgst

from mGST.simulate import apply_noisy_kraus_tensors_to_target_gate_set, get_random_noisy_kraus_tensors_for_gate_set

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Parameters

In [7]:
num_shots = 1000
seq_len_list = [1, 8, 14]
num_qubits = 2
dim = 2 ** num_qubits
state_rank = dim # max
povm_rank = dim # max
kraus_rank = 4 # start with a small rank
num_gates = 6
num_povm = dim
gate_set_name = "2QXYICZ"
noise_strength = 0.3

title_init = rf"{num_qubits}Q | $r_\mathrm{{Kraus}}$: {kraus_rank} | $r_\mathrm{{POVM}}$: {povm_rank} | $r_\mathrm{{State}}$: {state_rank} | Random Noisy Gate Set | Shots: {num_shots}"

# Tensors generations

In [8]:
target_operators, gate_labels = get_target_operators_gate_set_name(gate_set_name=gate_set_name)

random_kraus_tensors = get_random_noisy_kraus_tensors_for_gate_set(
    num_gates=num_gates,
    dim=dim,
    rank_kraus=kraus_rank,
    noise_strength=noise_strength,
    seed=42
)

noisy_gate_set_superop = apply_noisy_kraus_tensors_to_target_gate_set(
    kraus_tensors_noisy=random_kraus_tensors,
    target_gate_set=target_operators,
    is_target_vectorized=True,
)

kraus_tensor_noisy, povm_psd_noisy, state_psd_noisy = get_compressed_rep_from_mgst_output(
    *noisy_gate_set_superop.values(),
    kraus_rank=kraus_rank,
    state_rank=state_rank,
    povm_rank=povm_rank)

kraus_target_superop = kraus_tensor_to_mgst(target_operators["kraus"])

target_gate_set_superops = {
    "kraus": kraus_target_superop,
    "povm": target_operators["povm"],
    "state": target_operators["state"],
}

# Gathering them all in one place
noisy_gate_set_psd = {
    "kraus": kraus_tensor_noisy,
    "povm": povm_psd_noisy,
    "state": state_psd_noisy,
}